# Sesión 5 · Clasificación: días de alta y baja generación
**Curso:** Uso de IA para el análisis descriptivo y predictivo en energías renovables

> Ejecute cada celda con `Shift + Enter` y lea la explicación antes de cada paso. Todos los datos son **sintéticos** (simulados) para practicar.

**Objetivo:** entrenar tres clasificadores, leer la matriz de confusión y ver cómo el umbral cambia los errores.

## 1. Datos diarios
Simulamos 600 días con irradiancia media, temperatura, nubosidad y la energía generada. Definimos **alta** si supera la mediana.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
rng = np.random.default_rng(7)
n = 600
fechas = pd.date_range('2024-01-01', periods=n, freq='D')
nube = np.clip(rng.beta(2,3,n), 0, 1)
irr = 650*(1-0.8*nube) + rng.normal(0,40,n)
temp = 20 + 6*(1-nube) + rng.normal(0,1.5,n)
energia = 0.9*irr - 1.2*(temp-25) + rng.normal(0,35,n)
df = pd.DataFrame({'irr':irr,'temp':temp,'nube':nube,'energia':energia}, index=fechas)
df['alta'] = (df['energia'] > df['energia'].median()).astype(int)
print(df['alta'].value_counts())
df.head()

## 2. Dividir por fecha y escalar
Entrenamos con los primeros 480 días y probamos con los 120 últimos. El escalado se ajusta **solo con train**.

In [ ]:
from sklearn.preprocessing import StandardScaler
X = df[['irr','temp','nube']]; y = df['alta']
Xtr, Xte, ytr, yte = X.iloc[:480], X.iloc[480:], y.iloc[:480], y.iloc[480:]
esc = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = esc.transform(Xtr), esc.transform(Xte)
print(Xtr_s.shape, Xte_s.shape)

## 3. Tres clasificadores
Regresión logística, árbol de decisión (profundidad limitada) y kNN.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
modelos = {
  'Logística': LogisticRegression(),
  'Árbol (prof. 3)': DecisionTreeClassifier(max_depth=3, random_state=0),
  'kNN (k=7)': KNeighborsClassifier(n_neighbors=7),
}
for nombre, m in modelos.items():
    m.fit(Xtr_s, ytr); p = m.predict(Xte_s)
    print(f'{nombre:16s} exactitud {accuracy_score(yte,p):.2f} | precisión {precision_score(yte,p):.2f} | exhaustividad {recall_score(yte,p):.2f}')

## 4. Matriz de confusión
Filas: lo real. Columnas: lo predicho. La diagonal son los aciertos.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
log = modelos['Logística']
cm = confusion_matrix(yte, log.predict(Xte_s))
ConfusionMatrixDisplay(cm, display_labels=['baja','alta']).plot(cmap='Blues')
plt.show()
vn, fp, fn, vp = cm.ravel()
print('Falsos positivos:', fp, '| Falsos negativos:', fn)

## 5. Mover el umbral
La logística da una probabilidad. Por defecto decimos *alta* si pasa 0,5. Cambiemos el umbral y miremos los errores.

In [ ]:
prob = log.predict_proba(Xte_s)[:,1]
for u in [0.3, 0.5, 0.7]:
    p = (prob >= u).astype(int)
    vn, fp, fn, vp = confusion_matrix(yte, p).ravel()
    print(f'Umbral {u}: FP={fp:3d} FN={fn:3d} | precisión {precision_score(yte,p):.2f} | exhaustividad {recall_score(yte,p):.2f}')

**Lectura:** umbral bajo, más días marcados como altos: menos falsos negativos, más falsos positivos. Umbral alto, lo contrario. La decisión depende de qué error cuesta más.

## 6. Sobreajuste en el árbol
Probamos distintas profundidades y comparamos train y test.

In [ ]:
prof = range(1, 13)
a_tr, a_te = [], []
for d in prof:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr_s, ytr)
    a_tr.append(t.score(Xtr_s, ytr)); a_te.append(t.score(Xte_s, yte))
plt.plot(prof, a_tr, label='train'); plt.plot(prof, a_te, label='test')
plt.xlabel('max_depth'); plt.ylabel('exactitud'); plt.legend(); plt.show()

## 7. Ejercicios
1. Pruebe `k` = 1, 7 y 25 en kNN y compare.
2. ¿Qué umbral usaría si un falso negativo fuera 5 veces más costoso que un falso positivo?
3. Quite la variable `nube` y vea cómo cambia la exhaustividad.
4. ¿En qué profundidad empieza el sobreajuste del árbol?